# Fine-tune Mistral-7B-Instruct for the Strategy Agent (QLoRA) on Colab

This notebook runs the project's own training scripts (`strategy_agent/training/`) on a Colab GPU:

1. **Check the GPU**, load the project, install the libraries.
2. **Dry run** (no GPU used): checks the dataset against Mistral's real tokenizer.
3. **QLoRA SFT** of `mistralai/Mistral-7B-Instruct-v0.3` on the 1,200-example dataset (4-bit NF4, LoRA r=16, completion-only loss, sequence length 2048). Saves the **adapter only**.
4. **Evaluate** the fine-tuned model against the template generator and the rule baseline, on the same GPU.
5. *(Optional)* **DPO**, only if you have at least 50 real preference pairs.
6. **Download** the adapter and the report.

**Before you start:** Runtime > Change runtime type > **T4 GPU** (free) or a better one (L4 / A100, paid). Then run the cells top to bottom.

**Honest notes**
- This notebook was written and checked on a laptop, **not run on Colab by its author**. Expect to fix a small environment problem on the first run. The training code itself *has* been run (on CPU, tiny model); the 4-bit GPU run has not.
- - **Training refuses to start** if the prompt/answer alignment is wrong (see step 5), and prints how many examples it checked.
- **Time (estimates, not measurements):** on a free T4 expect roughly 2 to 4 hours for 2 epochs; on an A100 roughly 30 to 60 minutes for 3 epochs. Free sessions can disconnect; everything is saved to Google Drive and **re-running the SFT cell resumes from the last checkpoint**.
- The dataset labels come from the project's rule-based generator, so this teaches the **output format and constraint-following**, not new marketing insight. Measure it in step 4 before claiming anything.


## 1. Check the GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU found. Runtime > Change runtime type > T4 GPU (or better), then run this cell again."
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
CC = torch.cuda.get_device_capability(0)
print(f"{torch.cuda.get_device_name(0)}: {GPU_GB:.0f} GB, compute capability {CC[0]}.{CC[1]} -> training in {'bf16' if CC[0] >= 8 else 'fp16'} (a T4 is 7.5, so fp16)")

## 2. Settings (edit if you like)
`EPOCHS`: the brief says 2 to 3. On a 16 GB T4 use 2. If you run out of GPU memory, lower `SEQ_LEN` (examples up to about 2,000 tokens will then be truncated) or raise `GRAD_ACCUM`.


In [ ]:
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"     # public model, no access token needed
EPOCHS     = 2 if GPU_GB < 20 else 3
SEQ_LEN    = 2048
GRAD_ACCUM = 16                                        # effective batch = 1 x 16
USE_DRIVE  = True                                      # keep checkpoints and results on Google Drive so a disconnect does not lose them
print(dict(BASE_MODEL=BASE_MODEL, EPOCHS=EPOCHS, SEQ_LEN=SEQ_LEN, GRAD_ACCUM=GRAD_ACCUM, USE_DRIVE=USE_DRIVE))

## 3. Storage and the project
Upload `strategy_agent_colab.zip` (made for you) when asked. If you put it in `MyDrive/strategy_agent_run/` first, it is picked up automatically.

In [ ]:
import os, shutil, zipfile
os.chdir("/content")        # this cell deletes and recreates the project folder; do not be inside it (files.upload() saves into the current folder)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/strategy_agent_run"
else:
    PERSIST = "/content/strategy_agent_run"
os.makedirs(PERSIST, exist_ok=True)

ZIP = "strategy_agent_colab.zip"
src = next((p for p in (f"{PERSIST}/{ZIP}", f"/content/{ZIP}") if os.path.exists(p)), None)
if src is None:
    from google.colab import files
    print(f"Choose {ZIP} to upload ...")
    name = next(iter(files.upload()))
    src = f"/content/{name}"
PROJECT = "/content/strategy-agent"
shutil.rmtree(PROJECT, ignore_errors=True)
with zipfile.ZipFile(src) as z:
    z.extractall(PROJECT)
os.chdir(PROJECT)
print("project:", PROJECT, "| results will be saved to:", PERSIST)
print(sorted(os.listdir(".")))

## 4. Install the training libraries
These are the versions the training scripts were tested with. If pip asks you to **restart the session**, do (Runtime > Restart session), then run again from step 3: your Drive folder is kept.

In [ ]:
!pip install -q "trl==1.16.0" "peft==0.21.2" "transformers==5.20.2" "accelerate==1.15.0" "datasets==5.1.0" bitsandbytes
!pip check || true
import bitsandbytes, trl, peft, transformers
print("trl", trl.__version__, "| peft", peft.__version__, "| transformers", transformers.__version__, "| bitsandbytes", bitsandbytes.__version__)

## 5. Dry run (uses no GPU) and safety check
Loads Mistral's real tokenizer and the dataset and checks two things that would silently ruin a multi-hour run:
- **fit**: every example is at most `SEQ_LEN` tokens (none gets truncated);
- **alignment**: the tokenized *prompt* must be a prefix of the tokenized *prompt + answer*. If not, TRL trains the loss on the brief as well as the answer and on a context the model never sees at inference. (An earlier version of this project had exactly this bug with Mistral's chat template, which drops the system message once an answer is present; it is now folded into the user turn.)

This cell **stops with an error** if either check fails, so you do not start a long run by mistake.


In [ ]:
import subprocess, json
res = subprocess.run(["python", "-m", "strategy_agent.training.sft_qlora", "--dry-run", "--base-model", BASE_MODEL, "--seq-len", str(SEQ_LEN),
                      "--experiment-log", f"{PERSIST}/runs.jsonl"], capture_output=True, text=True)
text = res.stdout
assert "{" in text, "the dry run produced no summary:\n" + res.stdout[-1500:] + res.stderr[-1500:]
summary = json.loads(text[text.index("{"): text.rindex("}") + 1])
print(json.dumps(summary["recipe"], indent=1))
for split in ("train", "val"):
    d = summary["data"][split]
    print(split, {k: d[k] for k in ("examples", "max_tokens", "examples_over_seq_len", "fits")}, d["alignment"])
    assert d["fits"], f"{split}: some examples exceed SEQ_LEN={SEQ_LEN}; lower the dataset length or raise SEQ_LEN"
    assert d["alignment"]["aligned"], f"{split}: prompt/answer tokenisation is misaligned ({d['alignment']['prompt_prefix_mismatches']} examples); do NOT train"
print("OK: every example fits and is correctly aligned for", BASE_MODEL)

## 6. QLoRA fine-tuning (SFT)
This is the long step. The adapter is written to Drive. **If the session disconnects, reconnect, run steps 1 to 4 again, then run this cell again: it resumes from the last checkpoint.**
Watch `loss`: it should fall steadily. A loss that stays at the starting value, or `nan`, means something is wrong (see the troubleshooting table in `strategy_agent/training/RUN_ON_GPU.md`).


In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m strategy_agent.training.sft_qlora \
    --base-model {BASE_MODEL} --data-dir data/strategy_sft/v1 --out {PERSIST}/sft_adapter \
    --epochs {EPOCHS} --seq-len {SEQ_LEN} --grad-accum {GRAD_ACCUM} --resume --experiment-log {PERSIST}/runs.jsonl

### Check the result

In [ ]:
import json, pathlib
out = pathlib.Path(PERSIST) / "sft_adapter"
card = json.loads((out / "training_card.json").read_text())
print("final train loss:", card["final_train_loss"], "| steps:", card.get("data") and card["data"]["train"]["examples"], "training examples")
print("adapter files:", {p.name: f"{p.stat().st_size / 2**20:.0f} MB" for p in out.iterdir() if p.is_file()})
assert (out / "adapter_model.safetensors").exists() and (out / "adapter_config.json").exists(), "no adapter was saved"
cfg = json.loads((out / "adapter_config.json").read_text())
print("LoRA r / alpha / dropout:", cfg["r"], cfg["lora_alpha"], cfg["lora_dropout"], "| target modules:", cfg["target_modules"])

## 7. Evaluate the fine-tuned model
Installs the rest of the project's requirements, then runs the evaluation with the **fine-tuned Mistral (4-bit) against the template generator and the rule baseline**, on the same simulated world. `--llm-briefs 4` keeps it affordable on a T4; raise it for a tighter estimate.

**Read the "fell back to the template" column first.** If Mistral often writes invalid JSON, the service falls back to the template, and those results then measure the template, not the model.


In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
!python -m strategy_agent.eval.run --model mistral --adapter {PERSIST}/sft_adapter --load-4bit --llm-briefs 4 --out {PERSIST}/eval_reports

In [ ]:
import re, pathlib
md = (pathlib.Path(PERSIST) / "eval_reports" / "eval_report.md").read_text()
start = md.index("## Key findings"); end = md.index("## 1. Structural")
print(md[start:end])
i = md.index("## 5. Ablations")
print(md[i:i + 2500])

## 8. (Optional) DPO on real preference pairs
DPO needs **real preference pairs** from tenant feedback (`export_pairs_jsonl`), ideally 50 or more. **No such pairs exist yet**, so this cell does nothing unless you upload `pairs.jsonl` to `MyDrive/strategy_agent_run/`. It trains on top of the SFT adapter from step 6 (the reference is the SFT model).


In [ ]:
import os, subprocess
PAIRS = f"{PERSIST}/pairs.jsonl"
n = sum(1 for _ in open(PAIRS)) if os.path.exists(PAIRS) else 0
if n < 50:
    print(f"Skipping DPO: found {n} preference pairs (need about 50 or more real ones in {PAIRS}).")
else:
    subprocess.run(["python", "-m", "strategy_agent.training.dpo", "--base-model", BASE_MODEL, "--sft-adapter", f"{PERSIST}/sft_adapter",
                    "--pairs", PAIRS, "--out", f"{PERSIST}/dpo_adapter", "--resume", "--experiment-log", f"{PERSIST}/runs.jsonl"], check=True)

## 9. Download your results
Everything is already on Drive. This also zips the adapter and the experiment log so you can download them directly.

In [ ]:
import shutil
from google.colab import files
shutil.copy("experiments/runs.jsonl", f"{PERSIST}/runs_eval.jsonl") if os.path.exists("experiments/runs.jsonl") else None
zip_path = shutil.make_archive("/content/strategy_agent_results", "zip", PERSIST, "sft_adapter")
print("adapter zip:", zip_path)
files.download(zip_path)
print("Also on Drive:", sorted(os.listdir(PERSIST)))

## What to do with the results
- **Adapter:** put `sft_adapter/` on a machine that runs the service and set `LLM_MODEL_NAME=mistralai/Mistral-7B-Instruct-v0.3`, `ADAPTER_PATH=<the folder>`, `ADAPTER_ID=sft-v1`, `LLM_LOAD_4BIT=true` (needs a GPU to serve a 7B model).
- **Report:** open `eval_reports/eval_report.html` from Drive. Quote the **fine-tuned vs template vs rule baseline** numbers *with the fall-back rate*, and the small number of briefs.
- Anything surprising (loss not falling, mostly falling back to the template, out-of-memory errors): note the exact message and the cell it came from.
